In [ ]:
import pypsa
import os

In [ ]:
from _helpers_notebooks import mock_snakemake
snakemake = mock_snakemake(
        "plot_global_supply",
        scenario="default", # default, penalty-sa, penalty-ea, penalty-nwa, penalty-oc
        sort=True,
        demand="", #"__unreserved" or "" if reserved. Affects only the supply curve from single curves, not from the trade model
        wacc="regional",
        cost_year="2050",
        interone="hbi",
)

In [ ]:
n = pypsa.Network("../../resources/lco-hbi/cost_year~2050/wacc~regional/Europe_allocated_share/network_20-newre.nc")
n_old = pypsa.Network("../../resources/lco-hbi/cost_year~2050/wacc~regional/Europe_allocated_share/network_20-oldre.nc")
n_ea = pypsa.Network("../../resources/lco-hbi/cost_year~2050/wacc~regional/East_Asia_allocated_share/network_10.nc")

### Analysis

In [ ]:
n.generators.p_nom_max.div(1e3)

In [ ]:
n_old.generators.p_nom_max.div(1e3).sort_values(ascending=False)

In [ ]:
n.generators_t.p_max_pu.mean().sort_values(ascending=False)

In [ ]:
n_old.generators_t.p_max_pu.mean().sort_values(ascending=False)

In [ ]:
n_ea.links

In [ ]:
n.stores

In [ ]:
n.buses

In [ ]:
n.links

### Network adjustments

In [ ]:
n.add("Link", "link_bat_to_re", bus0="battery", bus1="renewable_electricity", p_nom_extendable=True, efficiency=1, capital_cost=1)

n.add("Link", "link_bat_to_bat", bus0="renewable_electricity", bus1="battery", p_nom_extendable=True, efficiency=1, capital_cost=1)

In [ ]:
# # Multiply p nom max by 10 in the network itself

n.generators.p_nom_max *= 10

In [ ]:
n.stores.loc["hbi_storage", "e_cyclic"] = True

### Solving of network

In [ ]:
config = snakemake.config

In [ ]:
solver_cfg = config.get("solver", {})
solver_name = os.getenv("SHIFT_SOLVER", solver_cfg.get("name", "glpk"))
solver_options_key = os.getenv(
    "SHIFT_SOLVER_OPTIONS", solver_cfg.get("options", "default")
)
solver_options = config.get("solver_options", {}).get(solver_options_key, {})

In [ ]:
status = n.optimize(
            n.snapshots,
            solver_name=solver_name,
            solver_options=solver_options,
            multi_investment_periods=False,
        )

In [ ]:
status

### Post solving analysis

In [ ]:
n.generators.p_nom_opt.div(1e3).round(2)

In [ ]:
n.generators.p_nom_opt.div(1e3).round(2)

In [ ]:
n.generators_t.p_max_pu.loc[:,"renewable_Europe_solar_6"][4000:4100].plot()

In [ ]:
n.generators_t.p_max_pu.loc[:,"renewable_Europe_solar_1"][4000:4100].plot()

In [ ]:
n.generators_t.p_max_pu.loc[:,"renewable_Europe_solar_1"].sort_values(ascending=False).reset_index(drop=True).plot()

In [ ]:
n.generators_t.p_max_pu.loc[:,"renewable_Europe_solar_6"].sort_values(ascending=False).reset_index(drop=True).plot()

In [ ]:
hbi_demand = n.statistics.withdrawal().to_frame().loc["Load", "hbi_demand"][0]
hbi_demand

In [ ]:
(n.statistics.system_cost() / hbi_demand).round(2)

In [ ]:
n.stores.e_nom_opt

In [ ]:
n.links

#### Stores

In [ ]:
n.stores_t.e["h2_storage"].plot()

In [ ]:
n.stores.e_initial

In [ ]:
n.stores.e_cyclic

### Bottom line and improvements

- Reestablish links from and to the battery
- Set hbi store `n.stores.loc["hbi_storage", "e_cyclic"] = True`